# Transformer Image Captioning — Local Training Notebook

End-to-end training notebook for the **Transformer** captioning model
(Vaswani et al., NeurIPS 2017) on Flickr30k — the second model in the
BUTD vs Transformer comparison study.

### Architecture
- **Encoder**: Pretrained ResNet-50 extracts a 7×7 spatial grid → 49 feature vectors of 2048-dim each
- **Decoder**: Transformer decoder with masked self-attention (text) + cross-attention (image patches)

### Pipeline overview
1. **Feature extraction** (one-time) — ResNet-50 saves `(49, 2048)` tensors to `data/transformer_features/`
2. **Training** — Transformer decoder trained on pre-extracted features with teacher forcing
3. **Evaluation** — BLEU-1/2/3/4 and METEOR on the validation split
4. **Sample predictions** — Qualitative inspection of generated captions

## 1. Setup

In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim

from src.data.splits import load_split_file, load_caption_map
from src.data.vocab import Vocabulary
from src.data.transformer_dataset import create_transformer_dataloader
from src.models.transformer_model import TransformerCaptionModel
from src.training.train_transformer import train_one_epoch, evaluate_loss, generate_captions
from src.eval.metrics import evaluate_captions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Compute device  : {device}")

## 2. Configuration

All hyperparameters are configurable here. The transformer architecture
parameters (`num_layers`, `num_heads`, `embed_dim`, `ff_dim`) can be
changed to run ablation experiments.

In [ ]:
# --- Paths ---
CAPTIONS_PATH  = Path("data/captions.txt")
IMAGES_DIR     = Path("data/Images")
FEATURES_DIR   = Path("data/transformer_features")
VOCAB_PATH     = Path("metadata/vocab/flickr30k_vocab.json")
SPLITS_DIR     = Path("metadata/splits")
CHECKPOINT_DIR = Path("checkpoints/transformer")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# --- Feature dimensions (fixed by ResNet-50 architecture) ---
NUM_PATCHES  = 49    # 7×7 spatial grid
FEATURE_DIM  = 2048  # ResNet-50 layer4 output channels

# --- Transformer hyperparameters (configurable) ---
EMBED_DIM  = 512   # d_model
NUM_HEADS  = 8     # must evenly divide EMBED_DIM
NUM_LAYERS = 3     # number of decoder layers
FF_DIM     = 2048  # feed-forward hidden dim
DROPOUT    = 0.1

# --- Training hyperparameters ---
BATCH_SIZE  = 32
LR          = 1e-4
GRAD_CLIP   = 1.0   # transformers need tighter clipping than LSTMs
NUM_EPOCHS  = 20
LR_STEP     = 5
LR_GAMMA    = 0.5

print("Configuration loaded.")

## 3. Feature Extraction (one-time)

Extracts a 7×7 = 49 spatial grid of 2048-dim features per image using
pretrained ResNet-50. Much faster than Faster R-CNN extraction.

**Estimated time: ~10–15 min CPU, ~2–3 min GPU for full Flickr30k.**

In [ ]:
train_ids = load_split_file(SPLITS_DIR / "train.txt")
val_ids   = load_split_file(SPLITS_DIR / "val.txt")
test_ids  = load_split_file(SPLITS_DIR / "test.txt")
all_ids   = train_ids + val_ids + test_ids

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
missing = [iid for iid in all_ids if not (FEATURES_DIR / f"{iid}.pt").exists()]
print(f"Images total   : {len(all_ids)}")
print(f"Features ready : {len(all_ids) - len(missing)}")
print(f"Missing        : {len(missing)}")

In [ ]:
if missing:
    print("Running feature extraction …")
    import subprocess
    result = subprocess.run(
        [
            sys.executable, "scripts/extract_transformer_features.py",
            "--images-dir", str(IMAGES_DIR),
            "--output-dir", str(FEATURES_DIR),
            "--splits-dir", str(SPLITS_DIR),
            "--batch-size", "16",
        ],
        capture_output=False,
    )
    if result.returncode != 0:
        raise RuntimeError("Feature extraction failed — check the output above.")
    print("Extraction complete.")
else:
    print("All features already extracted — skipping.")

In [ ]:
# Verify a feature file
sample_feat = torch.load(FEATURES_DIR / f"{all_ids[0]}.pt", weights_only=True)
print(f"Feature shape : {sample_feat.shape}")   # should be (49, 2048)
print(f"dtype         : {sample_feat.dtype}")   # float32

## 4. Dataset and DataLoaders

In [ ]:
vocab = Vocabulary.load(VOCAB_PATH)
vocab_size = len(vocab.word2idx)
print(f"Vocabulary size : {vocab_size}")

caption_map = load_caption_map(CAPTIONS_PATH)

train_loader = create_transformer_dataloader(
    caption_map=caption_map,
    features_dir=FEATURES_DIR,
    split_image_ids=train_ids,
    vocab=vocab,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = create_transformer_dataloader(
    caption_map=caption_map,
    features_dir=FEATURES_DIR,
    split_image_ids=val_ids,
    vocab=vocab,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Train batches : {len(train_loader)}  ({len(train_loader.dataset)} samples)")
print(f"Val batches   : {len(val_loader)}  ({len(val_loader.dataset)} samples)")

In [ ]:
# Sanity check batch shapes
sample_batch = next(iter(train_loader))
print("features     :", sample_batch["features"].shape)    # (B, 49, 2048)
print("caption_ids  :", sample_batch["caption_ids"].shape)  # (B, T)
print("target_ids   :", sample_batch["target_ids"].shape)   # (B, T)
print("Sample caption:", sample_batch["caption_texts"][0])

## 5. Model

The transformer hyperparameters are pulled from the configuration cell above.
To run an ablation, change `NUM_LAYERS`, `NUM_HEADS`, or `EMBED_DIM` in
Section 2 and re-run from this cell.

In [ ]:
model = TransformerCaptionModel(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FF_DIM,
    feature_dim=FEATURE_DIM,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print(f"Config: {NUM_LAYERS} layers, {NUM_HEADS} heads, embed_dim={EMBED_DIM}, ff_dim={FF_DIM}")

In [ ]:
# Forward-pass smoke test
model.eval()
with torch.no_grad():
    test_feats = sample_batch["features"][:2].to(device)
    test_caps  = sample_batch["caption_ids"][:2].to(device)
    logits = model(test_feats, test_caps)
    print(f"Logits shape : {logits.shape}  — expected (2, T, {vocab_size})")
    gen = model.generate(test_feats, vocab.word2idx["<start>"], vocab.word2idx["<end>"])
    print(f"Generated    : {' '.join(vocab.decode(gen[0].tolist()))}")
model.train();

## 6. Training

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)

train_losses = []
val_losses   = []
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, GRAD_CLIP)
    val_loss   = evaluate_loss(model, val_loader, criterion, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    flag = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "hparams": {
                    "vocab_size": vocab_size,
                    "embed_dim": EMBED_DIM,
                    "num_heads": NUM_HEADS,
                    "num_layers": NUM_LAYERS,
                    "ff_dim": FF_DIM,
                    "feature_dim": FEATURE_DIM,
                },
            },
            CHECKPOINT_DIR / "best.pt",
        )
        flag = "  ← best"

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS}  "
        f"train={train_loss:.4f}  "
        f"val={val_loss:.4f}{flag}"
    )

## 7. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
epochs = range(1, len(train_losses) + 1)
ax.plot(epochs, train_losses, label="Train", marker="o", markersize=4)
ax.plot(epochs, val_losses,   label="Val",   marker="s", markersize=4)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title(f"Transformer Training — Loss Curves ({NUM_LAYERS}L/{NUM_HEADS}H)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("notebooks/transformer_loss_curve.png", dpi=120)
plt.show()
print(f"Best val loss: {best_val_loss:.4f}")

## 8. Evaluation (BLEU / METEOR)

In [ ]:
checkpoint = torch.load(CHECKPOINT_DIR / "best.pt", map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}  (val_loss={checkpoint['val_loss']:.4f})")

print("Generating captions on validation split …")
val_predictions = generate_captions(model, val_loader, vocab, device)
print(f"Generated {len(val_predictions)} captions")

val_references = {iid: caption_map[iid] for iid in val_predictions if iid in caption_map}
scores = evaluate_captions(val_references, val_predictions, require_exact_match=False)

print("\n=== Validation Scores ===")
for metric, score in scores.items():
    print(f"  {metric.upper():8s}: {score:.4f}")

## 9. Sample Predictions

In [ ]:
import random
from PIL import Image

sample_ids = random.sample(sorted(val_predictions.keys()), min(6, len(val_predictions)))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, iid in zip(axes.flatten(), sample_ids):
    img_path = IMAGES_DIR / iid
    if img_path.exists():
        ax.imshow(Image.open(img_path).convert("RGB"))
    ax.axis("off")
    ax.set_title(val_predictions[iid], fontsize=7, wrap=True)
fig.suptitle("Transformer Generated Captions (val split)", fontsize=11)
fig.tight_layout()
fig.savefig("notebooks/transformer_sample_predictions.png", dpi=120)
plt.show()

In [ ]:
for iid in sample_ids:
    print(f"Image : {iid}")
    print(f"  Generated : {val_predictions[iid]}")
    print("  References:")
    for ref in caption_map.get(iid, []):
        print(f"    - {ref}")
    print()

## 10. Save Predictions

In [ ]:
import json

pred_path = Path("configs/transformer_val_predictions.json")
with pred_path.open("w", encoding="utf-8") as fh:
    json.dump(val_predictions, fh, indent=2)
print(f"Saved {len(val_predictions)} predictions to {pred_path}")
print("Run notebooks/results_analysis.ipynb to compare with BUTD.")